# Apply the synthetic-validation selection to the test split, once

Notebook 06 chose a scoring configuration on the synthetic-anomaly validation set (`results/improvements/synthetic_val/selected_settings.json`). This notebook reads that file and evaluates **exactly that configuration** on the official test split a single time, next to the current baseline, with paired-bootstrap 95% intervals. Nothing here is tuned: there is no loop over settings, and the notebook stops if the selection file says the test split was used to make it.

* **Baseline** = baseline predictors, `adaptive`, default `tau`, `lambda = 1`, **max** score (the row notebooks 04 and 05 call the baseline).
* **Selected** = the configuration in the JSON, with statistics refitted on **all** nominal validation images (per-position z-score, Mahalanobis, gate quantiles) or all training latents (channel standardisation), exactly as notebook 05 did.
* Because the selected `k` differs from the baseline's max score, the table also compares against the **baseline at the same `k`**, so a top-k effect is not credited to the scoring change.

A result with an interval that contains 0 is `inconclusive`, as in the earlier notebooks.

## 1. Configuration, splits, predictors and caches

In [1]:
import json
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml
from PIL import Image

try:
    display
except NameError:
    display = print

candidates = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next((p for p in candidates if (p / 'configs').is_dir() and (p / 'src').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run this notebook from the repository or its notebooks directory.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

CATEGORIES = ('bottle', 'capsule', 'hazelnut')
SELECTION_PATH = PROJECT_ROOT / 'results' / 'improvements' / 'synthetic_val' / 'selected_settings.json'
ENCODER_CHECKPOINT = PROJECT_ROOT / 'checkpoints' / 'IN22K-vit.g.16-600e.pth.tar'
BASELINE_CHECKPOINTS = PROJECT_ROOT / 'checkpoints' / 'predictors'
CACHE_ROOT = PROJECT_ROOT / 'data' / 'cache' / 'latents'
TEST_CACHE_ROOT = PROJECT_ROOT / 'data' / 'cache' / 'latents_test'
OUTPUT_ROOT = Path(os.environ.get('CONFIRMATION_OUTPUT', PROJECT_ROOT / 'results' / 'improvements' / 'confirmation'))
OVERWRITE = False
OUTPUT_FILES = ('selected_vs_baseline.csv', 'bootstrap_deltas.csv', 'confirmation_summary.json')
MASKING_CONFIG_PATH = PROJECT_ROOT / 'configs' / 'masking' / 'coarse_to_fine.yaml'
DATA_CONFIG = yaml.safe_load((PROJECT_ROOT / 'configs' / 'data' / 'mvtec.yaml').read_text(encoding='utf-8'))
VAL_FRACTION = float(DATA_CONFIG['nominal_split']['val_fraction'])
SPLIT_SEED = int(DATA_CONFIG['nominal_split']['seed'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_BOOTSTRAP, BOOTSTRAP_SEED = 1000, 0
# fitting choices that were declared (not tuned) in notebooks 05 and 06
POSITION_STD_FLOOR_FRACTION, CHANNEL_STD_EPS, MAHA_RIDGE = 0.5, 1e-6, 1e-3
TAU_QUANTILES = {'q0.9': 0.90, 'q0.95': 0.95, 'q0.99': 0.99, 'q0.995': 0.995}

if not OVERWRITE:
    existing = [OUTPUT_ROOT / n for n in OUTPUT_FILES if (OUTPUT_ROOT / n).exists()]
    if existing:
        raise FileExistsError('Refusing to overwrite: ' + ', '.join(map(str, existing)) + ' (set OVERWRITE = True)')

selection = json.loads(SELECTION_PATH.read_text(encoding='utf-8'))
if selection['test_used_for_selection'] is not False:
    raise ValueError('The selection file says the test split influenced the choice.')
SELECTED = selection['selected_settings']
print('selected settings (from notebook 06):', SELECTED)
SUPPORTED = {'baseline_cosine', 'pos_zscore', 'pos_centre', 'chan_standardise', 'layer_norm', 'maha_d32', 'maha_d64', 'maha_d128'}
if SELECTED['family'] not in SUPPORTED:
    raise ValueError(f'unsupported family {SELECTED["family"]!r}')

selected settings (from notebook 06): {'family': 'pos_zscore', 'k': 3, 'lam': 1.0, 'sigma': 0.0, 'tau': 'default'}


In [2]:
from ijepa_ad.caching.cache_schema import CacheExpectations, PreprocessingSpec
from ijepa_ad.caching.latent_cache import compute_file_sha256
from ijepa_ad.data.mvtec_dataset import validate_category_splits
from ijepa_ad.data.splits import split_nominal_train_val
from ijepa_ad.data.transforms import PreprocessConfig
from ijepa_ad.evaluation.metrics import aupro, image_auroc, pixel_auroc
from ijepa_ad.evaluation.runner import EMBEDDING_DIM, ENCODER_NAME, NUM_TOKENS, PATCH_SIZE
from ijepa_ad.inference import MaskedLatentScorer, load_masking_settings, load_predictor_checkpoint_with_provenance
from ijepa_ad.masking.score_maps import smooth_score_maps, upsample_patch_scores
from ijepa_ad.predictor.cache_dataset import load_cached_latent_dataset
from ijepa_ad.scoring import (
    ChannelStandardisedCosine, PositionNormaliser, fit_channel_stats, fit_mahalanobis_residual,
    layer_norm_cosine_discrepancy,
)

ENCODER_SHA256 = compute_file_sha256(ENCODER_CHECKPOINT)
MASKING_SETTINGS = load_masking_settings(MASKING_CONFIG_PATH)
DEFAULT_TAU, LAM_DEFAULT = MASKING_SETTINGS['threshold'], MASKING_SETTINGS['lam']
PREPROCESSING_SPEC = PreprocessingSpec.from_config(PreprocessConfig(image_size=int(DATA_CONFIG['preprocessing']['image_size'])))


def expectations(category, split, samples):
    return CacheExpectations(
        category=category, split=split, encoder_name=ENCODER_NAME, checkpoint_sha256=ENCODER_SHA256,
        patch_size=PATCH_SIZE, embedding_dim=EMBEDDING_DIM, preprocessing=PREPROCESSING_SPEC,
        sample_ids=tuple(s.image_id for s in samples), num_tokens=NUM_TOKENS,
    )


SPLITS, PREDICTORS, TRAIN, VAL, TESTSET = {}, {}, {}, {}, {}
for category in CATEGORIES:
    root = next(r for r in (PROJECT_ROOT / DATA_CONFIG['root'], PROJECT_ROOT / 'dataset', PROJECT_ROOT / 'dataset' / category)
                if (r / category / 'train').is_dir())
    train_ds, test_ds = validate_category_splits(root, category)
    nominal = split_nominal_train_val(
        train_ds.samples, val_fraction=VAL_FRACTION, seed=SPLIT_SEED,
        min_train=int(DATA_CONFIG['nominal_split']['min_train']), min_val=int(DATA_CONFIG['nominal_split']['min_val']),
    )
    TRAIN[category] = load_cached_latent_dataset(CACHE_ROOT, expectations(category, 'train', nominal.train), samples=train_ds.samples)
    VAL[category] = load_cached_latent_dataset(CACHE_ROOT, expectations(category, 'val', nominal.val), samples=train_ds.samples)
    TESTSET[category] = load_cached_latent_dataset(TEST_CACHE_ROOT, expectations(category, 'test', test_ds.samples), samples=test_ds.samples)
    PREDICTORS[category], _ = load_predictor_checkpoint_with_provenance(
        BASELINE_CHECKPOINTS / f'{category}_cosine.pt', category=category, encoder_sha256=ENCODER_SHA256, device=DEVICE)
    print(f'{category}: train={len(TRAIN[category])}, val={len(VAL[category])}, test={len(TESTSET[category])}')

bottle: train=188, val=21, test=83
capsule: train=197, val=22, test=132
hazelnut: train=352, val=39, test=110


## 2. Fit the statistics on nominal data, then score the test split

`run_scorer` is the loop of notebooks 04 to 06. The fine-pass gate is evaluated on the baseline `1 - cos` region scores, so the selected configuration evaluates the same windows as the baseline unless the selected `tau` differs from the default (the printed `masks/image` says which).

In [3]:
def run_scorer(predictor, dataset, mode, threshold=DEFAULT_TAU, *, discrepancy=None, timed=False):
    scorer = MaskedLatentScorer(predictor, discrepancy=discrepancy, **{**MASKING_SETTINGS, 'threshold': float(threshold)})
    if timed:
        scorer.score(dataset[0].latent.unsqueeze(0).to(DEVICE), mode=mode)   # warm-up
    coarse, fine, work, seconds = [], [], [], 0.0
    for sample in dataset:
        latent = sample.latent.unsqueeze(0).to(DEVICE)
        if timed and DEVICE == 'cuda':
            torch.cuda.synchronize()
        started = time.perf_counter()
        result = scorer.score(latent, mode=mode)
        if timed and DEVICE == 'cuda':
            torch.cuda.synchronize()
        seconds += time.perf_counter() - started
        coarse.append(result.coarse_scores.detach().cpu()); fine.append(result.fine_scores.detach().cpu()); work.append(result.evaluated_regions)
    return {'coarse': torch.stack(coarse), 'fine': torch.stack(fine), 'work': np.asarray(work), 'ms': 1000 * seconds / len(dataset), 'scorer': scorer}


def region_scores(scorer, coarse):
    out = torch.zeros_like(coarse)
    for window in scorer.coarse.windows():
        idx = list(window.token_indices)
        out[:, idx] = coarse[:, idx].mean(dim=1, keepdim=True)
    return out


def load_ground_truth(annotation):
    if annotation.mask_path is None:
        with Image.open(annotation.image_path) as image:
            width, height = image.size
        return np.zeros((height, width), dtype=bool)
    with Image.open(annotation.mask_path) as mask_image:
        return np.asarray(mask_image.convert('L')) > 0


def selected_tau(category):
    key = SELECTED['tau']
    if key == 'default':
        return DEFAULT_TAU
    run = run_scorer(PREDICTORS[category], VAL[category], 'fixed')
    region = region_scores(run['scorer'], run['coarse'])
    return float(np.quantile(region.flatten().numpy(), TAU_QUANTILES[key]))


CONFIGS, TEST = {}, {}
for category in CATEGORIES:
    started = time.perf_counter()
    predictor, family, lam = PREDICTORS[category], SELECTED['family'], float(SELECTED['lam'])
    dataset = TESTSET[category]
    TEST[category] = {
        'labels': np.asarray([int(s.annotation.label != 'good') for s in dataset]),
        'masks': [load_ground_truth(s.annotation) for s in dataset],
    }
    # ---- baseline: adaptive, default tau, cosine ----
    base = run_scorer(predictor, dataset, 'adaptive', timed=True)
    n_coarse = len(base['scorer'].coarse.windows())
    CONFIGS[(category, 'baseline')] = {'patch': base['coarse'] + LAM_DEFAULT * base['fine'], 'work': base['work'], 'ms': base['ms'], 'sigma': 0.0, 'k': None, 'n_coarse': n_coarse}

    # ---- selected configuration ----
    tau = selected_tau(category)
    gate_run = base if tau == DEFAULT_TAU else run_scorer(predictor, dataset, 'adaptive', tau, timed=True)
    if family in ('baseline_cosine', 'pos_zscore', 'pos_centre'):
        coarse, fine, work, ms = gate_run['coarse'], gate_run['fine'], gate_run['work'], gate_run['ms']
        if family != 'baseline_cosine':
            val_c = run_scorer(predictor, VAL[category], 'fixed')['coarse']
            val_f = run_scorer(predictor, VAL[category], 'dense')['fine']
            kwargs = {'divide': True, 'std_floor_fraction': POSITION_STD_FLOOR_FRACTION} if family == 'pos_zscore' else {'divide': False}
            norm_c, norm_f = PositionNormaliser.fit(val_c, **kwargs), PositionNormaliser.fit(val_f, **kwargs)
            fired = region_scores(gate_run['scorer'], gate_run['coarse']) >= tau
            coarse, fine = norm_c.apply(coarse), fired.float() * norm_f.apply(fine)
    else:
        if family == 'chan_standardise':
            discrepancy = ChannelStandardisedCosine(fit_channel_stats(TRAIN[category].latents, eps=CHANNEL_STD_EPS))
        elif family == 'layer_norm':
            discrepancy = layer_norm_cosine_discrepancy
        else:
            scorer0 = MaskedLatentScorer(predictor, **MASKING_SETTINGS)
            residuals = torch.cat([
                (s.latent.unsqueeze(0).to(DEVICE).float() - scorer0.window_predictions(s.latent.unsqueeze(0).to(DEVICE), scorer0.coarse.windows()).float()).cpu()[0]
                for s in VAL[category]
            ])
            discrepancy = fit_mahalanobis_residual(residuals, n_components=int(family[6:]), ridge=MAHA_RIDGE)
        run = run_scorer(predictor, dataset, 'adaptive', tau, discrepancy=discrepancy, timed=True)
        coarse, fine, work, ms = run['coarse'], run['fine'], run['work'], run['ms']
    CONFIGS[(category, 'selected')] = {'patch': coarse + lam * fine, 'work': work, 'ms': ms, 'sigma': float(SELECTED['sigma']), 'k': int(SELECTED['k']), 'n_coarse': n_coarse}
    print(f'{category}: scored in {time.perf_counter() - started:.0f}s; tau={tau:.3f}; masks/image baseline {base["work"].mean():.2f}, selected {work.mean():.2f}')

C:\Users\aryan\OneDrive\Desktop\7 sem\I-JEPA-AD\.venv\Lib\site-packages\torch\nn\modules\transformer.py:685: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  return torch._transformer_encoder_layer_fwd(
bottle: scored in 7s; tau=0.500; masks/image baseline 16.19, selected 16.19
capsule: scored in 9s; tau=0.500; masks/image baseline 16.00, selected 16.00
hazelnut: scored in 13s; tau=0.500; masks/image baseline 16.15, selected 16.15


## 3. Metrics and paired bootstrap

Same definitions as notebook 05: image score from the token scores (smoothed maps are read at the token centres), pixel maps upsampled straight to the ground-truth size when `sigma = 0` (otherwise 14x14 to 224x224, blur, then to ground-truth size), AUPRO with 200 thresholds. The bootstrap resamples good and anomalous test images separately, 1,000 times, with the same resample for every comparison (seed 0, the stream of notebooks 04 and 05).

In [4]:
def centre_samples(maps224):
    return F.avg_pool2d(maps224[:, :, 7:, 7:], kernel_size=2, stride=16).flatten(1)


def evaluate(category, config):
    labels, masks, patch, sigma = TEST[category]['labels'], TEST[category]['masks'], config['patch'], config['sigma']
    if sigma == 0.0:
        token_scores = patch
        maps = [upsample_patch_scores(p, output_size=m.shape)[0, 0].numpy().astype(np.float32) for p, m in zip(patch, masks)]
    else:
        smoothed = smooth_score_maps(upsample_patch_scores(patch, output_size=224), sigma)
        token_scores = centre_samples(smoothed)
        maps = [F.interpolate(smoothed[i:i + 1], size=m.shape, mode='bilinear', align_corners=False)[0, 0].numpy().astype(np.float32) for i, m in enumerate(masks)]
    k = int(SELECTED['k'])
    scores = {'max': token_scores.max(dim=1).values.numpy(), f'top{k}': token_scores.topk(k, dim=1).values.mean(dim=1).numpy()}
    return {'scores': scores, 'pixel_auroc': pixel_auroc(maps, masks), 'aupro': aupro(maps, masks, num_thresholds=200)}


K = int(SELECTED['k'])
RESULTS = {}
for category in CATEGORIES:
    for name in ('baseline', 'selected'):
        RESULTS[(category, name)] = evaluate(category, CONFIGS[(category, name)])
        # the baseline's top-k scores are needed for the like-for-like reference
    print(category, 'evaluated')

rows = []
for category in CATEGORIES:
    labels = TEST[category]['labels']
    for name in ('baseline', 'selected'):
        r, c = RESULTS[(category, name)], CONFIGS[(category, name)]
        rows.append({
            'category': category, 'config': name, 'image_auroc_max': image_auroc(labels, r['scores']['max']),
            f'image_auroc_top{K}': image_auroc(labels, r['scores'][f'top{K}']), 'pixel_auroc': r['pixel_auroc'], 'aupro': r['aupro'],
            'masks_per_image': float(c['work'].mean()), 'scorer_ms_per_image': c['ms'],
        })
metrics_df = pd.DataFrame(rows)
display(metrics_df.round(4))

bottle evaluated
capsule evaluated
hazelnut evaluated


,category,config,image_auroc_max,image_auroc_top3,pixel_auroc,aupro,masks_per_image,scorer_ms_per_image
0,bottle,baseline,0.9246,0.9516,0.9295,0.7657,16.1928,22.4659
1,bottle,selected,0.9929,0.9976,0.9524,0.8309,16.1928,22.4659
2,capsule,baseline,0.5217,0.5122,0.6845,0.0981,16.0000,23.0953
3,capsule,selected,0.6223,0.6721,0.9011,0.5080,16.0000,23.0953
4,hazelnut,baseline,0.8021,0.8975,0.9067,0.5774,16.1455,23.5091
5,hazelnut,selected,0.8643,0.9393,0.9692,0.8118,16.1455,23.5091


In [5]:
def bootstrap_indices(labels):
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    pos, neg = np.flatnonzero(labels == 1), np.flatnonzero(labels == 0)
    return [np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))]) for _ in range(N_BOOTSTRAP)]


def verdict(low, high):
    return 'improves' if low > 0 else 'worsens' if high < 0 else 'inconclusive (CI contains 0)'


boot_rows = []
for category in CATEGORIES:
    labels = TEST[category]['labels']
    indices = bootstrap_indices(labels)
    s, b = RESULTS[(category, 'selected')]['scores'], RESULTS[(category, 'baseline')]['scores']
    comparisons = (
        (f'selected, top-{K} score', s[f'top{K}'], 'baseline, max score (the requested reference)', b['max']),
        (f'selected, top-{K} score', s[f'top{K}'], f'baseline, top-{K} score (same score)', b[f'top{K}']),
        ('selected, max score', s['max'], 'baseline, max score', b['max']),
    )
    for new_label, new, ref_label, ref in comparisons:
        new_boot = np.asarray([image_auroc(labels[i], new[i]) for i in indices])
        ref_boot = np.asarray([image_auroc(labels[i], ref[i]) for i in indices])
        low, high = np.percentile(new_boot - ref_boot, [2.5, 97.5])
        boot_rows.append({
            'category': category, 'comparison': new_label, 'reference': ref_label,
            'image_auroc': image_auroc(labels, new), 'reference_auroc': image_auroc(labels, ref),
            'delta_image_auroc': image_auroc(labels, new) - image_auroc(labels, ref), 'ci95_low': low, 'ci95_high': high, 'verdict': verdict(low, high),
        })
boot_df = pd.DataFrame(boot_rows)
shown = boot_df.copy()
shown['CI95'] = shown.apply(lambda r: f'[{r.ci95_low:+.3f}, {r.ci95_high:+.3f}]', axis=1)
display(shown[['category', 'comparison', 'reference', 'image_auroc', 'delta_image_auroc', 'CI95', 'verdict']].round(4))

,category,comparison,reference,image_auroc,delta_image_auroc,CI95,verdict
0,bottle,"selected, top-3 score","baseline, max score (the requested reference)",0.9976,0.0730,"[+0.019, +0.152]",improves
1,bottle,"selected, top-3 score","baseline, top-3 score (same score)",0.9976,0.0460,"[+0.012, +0.097]",improves
2,bottle,"selected, max score","baseline, max score",0.9929,0.0683,"[+0.013, +0.145]",improves
3,capsule,"selected, top-3 score","baseline, max score (the requested reference)",0.6721,0.1504,"[-0.045, +0.356]",inconclusive (CI contains 0)
4,capsule,"selected, top-3 score","baseline, top-3 score (same score)",0.6721,0.1600,"[-0.027, +0.357]",inconclusive (CI contains 0)
5,capsule,"selected, max score","baseline, max score",0.6223,0.1005,"[-0.110, +0.318]",inconclusive (CI contains 0)
6,hazelnut,"selected, top-3 score","baseline, max score (the requested reference)",0.9393,0.1371,"[+0.054, +0.226]",improves
7,hazelnut,"selected, top-3 score","baseline, top-3 score (same score)",0.9393,0.0418,"[-0.014, +0.104]",inconclusive (CI contains 0)
8,hazelnut,"selected, max score","baseline, max score",0.8643,0.0621,"[-0.037, +0.168]",inconclusive (CI contains 0)


### Consistency with notebook 05

When the selected configuration is a notebook 05 variant (here `pos_zscore` with `sigma = 0`), its pixel AUROC and AUPRO must equal that variant's saved row, because the pixel maps do not depend on `k`. A mismatch would mean this notebook did not evaluate what notebook 05 did.

In [6]:
CONSISTENT = None
tier1 = PROJECT_ROOT / 'results' / 'improvements' / 'tier1' / 'variant_metrics.csv'
name05 = {'baseline_cosine': 'baseline', 'pos_zscore': '1a_position_zscore', 'pos_centre': '1b_position_centre_only',
          'chan_standardise': '2a_channel_standardise', 'layer_norm': '2b_layer_norm', 'maha_d32': '4_mahalanobis_d32',
          'maha_d64': '4_mahalanobis_d64', 'maha_d128': '4_mahalanobis_d128'}[SELECTED['family']]
if tier1.exists() and SELECTED['sigma'] == 0.0 and SELECTED['tau'] == 'default' and float(SELECTED['lam']) == LAM_DEFAULT:
    saved = pd.read_csv(tier1)
    diffs = []
    for category in CATEGORIES:
        row = saved[(saved.category == category) & (saved.variant == name05)].iloc[0]
        mine = metrics_df[(metrics_df.category == category) & (metrics_df.config == 'selected')].iloc[0]
        diffs.append({'category': category, 'd_pixel_auroc': abs(mine.pixel_auroc - row.pixel_auroc), 'd_aupro': abs(mine.aupro - row.aupro),
                      'd_image_auroc_max': abs(mine.image_auroc_max - row.image_auroc_max)})
    diff_df = pd.DataFrame(diffs)
    display(diff_df)
    CONSISTENT = bool((diff_df[['d_pixel_auroc', 'd_aupro', 'd_image_auroc_max']] < 1e-4).all().all())
    print('matches notebook 05 variant', name05, ':', CONSISTENT)
    if not CONSISTENT:
        raise AssertionError('The selected configuration does not reproduce its notebook 05 row.')

matches notebook 05 variant 1a_position_zscore : True


,category,d_pixel_auroc,d_aupro,d_image_auroc_max
0,bottle,0.000000e+00,0.0,1.110223e-16
1,capsule,1.110223e-16,0.0,0.000000e+00
2,hazelnut,0.000000e+00,0.0,0.000000e+00


## Save and read the result

In [7]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if not OVERWRITE:
    existing = [OUTPUT_ROOT / n for n in OUTPUT_FILES if (OUTPUT_ROOT / n).exists()]
    if existing:
        raise FileExistsError('Refusing to overwrite: ' + ', '.join(map(str, existing)))
metrics_df.to_csv(OUTPUT_ROOT / 'selected_vs_baseline.csv', index=False)
boot_df.to_csv(OUTPUT_ROOT / 'bootstrap_deltas.csv', index=False)
(OUTPUT_ROOT / 'confirmation_summary.json').write_text(json.dumps({
    'selection_file': str(SELECTION_PATH.relative_to(PROJECT_ROOT)),
    'selected_settings': SELECTED,
    'chosen_from': selection['chosen_from'],
    'test_used_for_selection': False,
    'test_split_scored': 'once, for the baseline and the selected configuration only',
    'statistics_refitted_on': 'all nominal validation images (position, Mahalanobis, gate quantile) / all training latents (channel standardisation)',
    'bootstrap': {'n': N_BOOTSTRAP, 'seed': BOOTSTRAP_SEED},
    'reproduces_notebook05_variant': CONSISTENT,
    'verdicts_vs_baseline_max': boot_df[boot_df.reference.str.contains('requested')][['category', 'verdict']].set_index('category').verdict.to_dict(),
}, indent=2, sort_keys=True, default=str), encoding='utf-8')
print('saved to', OUTPUT_ROOT)

saved to C:\Users\aryan\OneDrive\Desktop\7 sem\I-JEPA-AD\results\improvements\confirmation


## What you can and cannot say

**You can say**

* The configuration was chosen on synthetic defects made from nominal validation images, before this test evaluation, and the test split was scored once for it.
* The change in image AUROC against the baseline, with a paired-bootstrap interval, both against the baseline's max score and against the baseline at the same `k`.
* Pixel AUROC, AUPRO and masks per image for both configurations (pixel metrics have no interval).

**You cannot say**

* That an interval containing 0 shows an improvement or a harm.
* That this configuration is the best possible: only the options declared in notebook 06 were considered, and ties were broken toward the simpler option.
* Anything about other categories, other seeds, or a different encoder layer or mask size.